# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Imtiyazsoomro/flyrank-ml-internship-imtiyaz/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [4]:
import os
import pandas as pd
import numpy as np

# 1. Environment and Data Setup
if not os.path.exists('data/raw/content_refresh_anonymized.csv'):
    !git clone https://github.com/imtiyazsoomro/flyrank-ml-internship-imtiyaz.git
    %cd flyrank-ml-internship-imtiyaz

df = pd.read_csv('data/raw/content_refresh_anonymized.csv')

# 2. Define the baseline scoring rule function
def assign_baseline_rule(row):
    if row['days_since_last_update'] > 180 and row['impressions_90d'] > 500:
        return 'REFRESH_PRIORITY', 'STALE_HIGH_VISIBILITY', (row['days_since_last_update'] / 365) * np.log1p(row['impressions_90d'])
    elif row['days_since_last_update'] > 180:
        return 'MAINTAIN', 'STALE_LOW_VISIBILITY', (row['days_since_last_update'] / 365)
    else:
        return 'NO_ACTION', 'FRESH_ACTIVE', 0.0

# Apply rule to dataframe
results = df.apply(assign_baseline_rule, axis=1)
df['action'] = [r[0] for r in results]
df['reason_code'] = [r[1] for r in results]
df['action_score'] = [r[2] for r in results]

print("Baseline rule evaluation summary:")
print(df['action'].value_counts())

Baseline rule evaluation summary:
action
NO_ACTION           29826
MAINTAIN              157
REFRESH_PRIORITY       17
Name: count, dtype: int64


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [5]:
# Create outputs folder if it does not exist
os.makedirs('work/outputs', exist_ok=True)

# Sort by action score descending to build the priority queue
ranked_queue = df.sort_values(by='action_score', ascending=False).reset_index(drop=True)

# Select key columns for the export artifact
export_cols = ['days_since_last_update', 'impressions_90d', 'avg_position', 'action', 'reason_code', 'action_score', 'trend_direction']
output_path = 'work/outputs/baseline_action_score.csv'

# Save the CSV artifact
ranked_queue[export_cols].to_csv(output_path, index=False)

print(f"Ranked queue successfully generated with {len(ranked_queue):,} rows.")
print(f"Saved artifact to: {output_path}")

Ranked queue successfully generated with 30,000 rows.
Saved artifact to: work/outputs/baseline_action_score.csv


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [6]:
# Extract and display top 20 queue items
top_20 = ranked_queue.head(20)[['days_since_last_update', 'impressions_90d', 'avg_position', 'action', 'reason_code', 'action_score', 'trend_direction']]

print("=== TOP 20 BASELINE QUEUE REVIEW ===")
print(top_20.to_string())

# Summary analysis of top 20 accuracy
declining_in_top20 = (top_20['trend_direction'] == 'down').sum()
print(f"\nPrecision Audit: {declining_in_top20}/20 top items are actively declining in traffic.")

=== TOP 20 BASELINE QUEUE REVIEW ===
    days_since_last_update  impressions_90d  avg_position            action            reason_code  action_score trend_direction
0                      194            61678          19.7  REFRESH_PRIORITY  STALE_HIGH_VISIBILITY      5.862360            down
1                      194            59472          24.8  REFRESH_PRIORITY  STALE_HIGH_VISIBILITY      5.843002            down
2                      301              954           9.0  REFRESH_PRIORITY  STALE_HIGH_VISIBILITY      5.658562            down
3                      301              821           5.8  REFRESH_PRIORITY  STALE_HIGH_VISIBILITY      5.534887            down
4                      194            25715          22.2  REFRESH_PRIORITY  STALE_HIGH_VISIBILITY      5.397382            down
5                      193            13299          10.5  REFRESH_PRIORITY  STALE_HIGH_VISIBILITY      5.020918            down
6                      194             7812          39.0  R

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [7]:
# Identify potential weak picks (Top ranked pages that are actually STABLE/UP, not declining)
weak_picks = top_20[top_20['trend_direction'] != 'down']

print(f"Identified {len(weak_picks)} weak picks in the top 20 (Evergreen / False Positives):")
print(weak_picks[['days_since_last_update', 'impressions_90d', 'avg_position', 'trend_direction']])

# Leakage assertion check
assert 'trend_pct' not in df.columns or df['action_score'].corr(df['trend_pct']) < 0.95, "Leakage check failed!"
print("\nLeakage Check Passed: Rule score is built purely on observable signals.")

Identified 2 weak picks in the top 20 (Evergreen / False Positives):
    days_since_last_update  impressions_90d  avg_position trend_direction
12                     194             1316          21.8          stable
17                     373                1           1.0            flat

Leakage Check Passed: Rule score is built purely on observable signals.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.